In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, date_format

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# DIM CLIENTES
df_dim_clientes = (
    spark.table("silver.clientes")
    .select("id_cliente", "nombre", "pais", "email")
    .dropDuplicates(["id_cliente"])
)
df_dim_clientes.write.mode("overwrite").saveAsTable("gold.dim_clientes")

# DIM PRODUCTOS
df_dim_productos = (
    spark.table("silver.productos")
    .select("id_producto", "nombre_producto", "categoria", "precio_unitario")
    .dropDuplicates(["id_producto"])
)
df_dim_productos.write.mode("overwrite").saveAsTable("gold.dim_productos")

# DIM FECHA
df_dim_fecha = (
    spark.table("silver.ordenes")
    .select("fecha_orden")
    .dropDuplicates(["fecha_orden"])
    .withColumn("anio", year(col("fecha_orden")))
    .withColumn("mes", month(col("fecha_orden")))
    .withColumn("dia", dayofmonth(col("fecha_orden")))
    .withColumn("mes_nombre", date_format(col("fecha_orden"), "MMMM"))
)
df_dim_fecha.write.mode("overwrite").saveAsTable("gold.dim_fecha")

# FACT VENTAS
df_fact_ventas = (
    spark.table("silver.detalle_orden").alias("d")
    .join(
        spark.table("silver.ordenes").alias("o"),
        col("d.id_orden") == col("o.id_orden"),
        "inner"
    )
    .select(
        col("d.id_detalle"),
        col("d.id_orden"),
        col("o.id_cliente"),
        col("d.id_producto"),
        col("o.fecha_orden"),
        col("o.estado_pago"),
        col("d.cantidad"),
        col("d.precio_unitario"),
        col("d.monto_linea")
    )
)

df_fact_ventas.write.mode("overwrite").saveAsTable("gold.fact_ventas")

spark.sql("SHOW TABLES IN gold").show()

In [0]:
spark.sql("SHOW TABLES IN gold").show()

In [0]:
spark.table("raw.clientes").count()
spark.table("raw.productos").count()
spark.table("raw.ordenes").count()
spark.table("raw.detalle_orden").count()

In [0]:
capas = {
    "raw": ["clientes", "productos", "ordenes", "detalle_orden"],
    "bronze": ["clientes", "productos", "ordenes", "detalle_orden"],
    "silver": ["clientes", "productos", "ordenes", "detalle_orden"],
    "gold": ["dim_clientes", "dim_productos", "dim_fecha", "fact_ventas"]
}

for esquema, tablas in capas.items():
    print(f"\n===== ESQUEMA: {esquema.upper()} =====")
    for tabla in tablas:
        nombre = f"{esquema}.{tabla}"
        try:
            cantidad = spark.table(nombre).count()
            print(f"{nombre}: {cantidad} registros")
        except Exception as e:
            print(f"{nombre}: ERROR / NO EXISTE")

In [0]:
%sql
SELECT * FROM gold.dim_clientes LIMIT 10